# Урок 30 — PostgreSQL: Архітектура та Практика

> **Вимоги:** Docker-контейнер `course-postgres` запущений (`docker compose up -d postgres`)
> 
> **Бібліотека:** `psycopg2-binary` — офіційний Python-драйвер для PostgreSQL

```bash
# Встановлення (якщо не встановлено):
pip install psycopg2-binary
```

---

## Що ми вивчимо

| Розділ | Тема |
|--------|------|
| 1 | PostgreSQL vs SQLite — ключові відмінності |
| 2 | Підключення через psycopg2 |
| 3 | PostgreSQL-специфічні типи даних |
| 4 | DDL, DML — нові можливості |
| 5 | EXPLAIN ANALYZE — заглядаємо під капот |
| 6 | Індекси: B-Tree, GIN |
| 7 | JSONB — NoSQL всередині PostgreSQL |
| 8 | Вікончаті функції (Window Functions) |
| 9 | CTE та рекурсивні запити |
| 10 | Транзакції та рівні ізоляції |


---
## 1. PostgreSQL vs SQLite — ключові відмінності

| Характеристика | SQLite | PostgreSQL |
|----------------|--------|------------|
| Архітектура | Файлова (один файл) | Клієнт-серверна (daemon процес) |
| Конкурентність | 1 writer, багато readers | Повна багатопоточність (MVCC) |
| Типи даних | Динамічна типізація | Строга статична типізація |
| JSONB | Немає | Повна підтримка з індексами |
| Full-Text Search | Обмежений | Вбудований (GIN/GiST) |
| Window Functions | Обмежені | Повна підтримка |
| Partition | Немає | Вбудований |
| Масиви | Немає | Нативні типи ARRAY |
| Реплікація | Немає | Streaming replication |
| Обмеження розміру | 281 ТБ | Необмежено |
| DCL (GRANT/REVOKE) | Немає | Повна підтримка |

### OLTP vs OLAP

PostgreSQL — **рядкова БД (row-store)**, оптимізована для **OLTP** (Online Transaction Processing):
- Дані одного рядка зберігаються поруч на диску
- Ідеально для INSERT/UPDATE/DELETE окремих записів
- Тисячі паралельних транзакцій без блокувань (MVCC)

**Слабке місце:** аналітичні запити по одному стовпцю (aggregation over millions of rows) — доводиться зчитувати весь рядок. Для цього є колонкові БД (DuckDB, ClickHouse, Redshift).

---
## 2. Підключення через psycopg2

In [ ]:
import psycopg2
from psycopg2.extras import RealDictCursor, Json
import json
from datetime import datetime, date

# ---- Параметри підключення до Docker-контейнера ----
# Ці параметри відповідають docker-compose.yml
DB_CONFIG = {
    'host':     'localhost',   # контейнер пробрасує порт 5432 на localhost
    'port':     5432,
    'dbname':   'course_db',
    'user':     'student',
    'password': 'python2024',
}

# connect() відкриває TCP-з'єднання до PostgreSQL сервера
# autocommit=False (за замовчуванням) — транзакції явні
conn = psycopg2.connect(**DB_CONFIG)

# RealDictCursor: рядки повертаються як dict {'column': value}
# замість tuple (0, 'Alice', ...) як у стандартному cursor
cursor = conn.cursor(cursor_factory=RealDictCursor)

print('Підключено до PostgreSQL!')

# Перевіряємо версію сервера
cursor.execute('SELECT version()')
print(cursor.fetchone()['version'])

---
## 3. PostgreSQL-специфічні типи даних

### Числові типи
| Тип | Розмір | Діапазон |
|-----|--------|----------|
| `SMALLINT` | 2 байти | -32768 .. 32767 |
| `INTEGER` | 4 байти | -2B .. 2B |
| `BIGINT` | 8 байтів | -9.2e18 .. 9.2e18 |
| `NUMERIC(p,s)` | змінний | точна арифметика (фінанси!) |
| `REAL` | 4 байти | 6 знаків точності |
| `DOUBLE PRECISION` | 8 байтів | 15 знаків точності |

### Спеціальні типи (недоступні в SQLite)
| Тип | Призначення |
|-----|-------------|
| `SERIAL` / `BIGSERIAL` | Авто-інкремент (насправді sequence + INTEGER) |
| `UUID` | 128-бітний унікальний ідентифікатор |
| `JSONB` | JSON у бінарному форматі (індексований!) |
| `ARRAY` | Масив будь-якого типу: `INTEGER[]`, `TEXT[]` |
| `TIMESTAMPTZ` | Мітка часу з часовим поясом |
| `INTERVAL` | Тривалість часу: `'3 months 5 days'` |
| `INET` | IP-адреса з валідацією |
| `TSVECTOR` | Full-text search вектор |
| `GEOMETRY` | Геопросторові дані (з PostGIS) |

---
## 4. DDL — Створення схеми

In [ ]:
# ---- Очищуємо від попередніх запусків (DROP IF EXISTS) ----
# CASCADE видаляє залежні об'єкти разом з таблицею
cleanup_sql = """
    DROP TABLE IF EXISTS order_items CASCADE;
    DROP TABLE IF EXISTS orders     CASCADE;
    DROP TABLE IF EXISTS products   CASCADE;
    DROP TABLE IF EXISTS customers  CASCADE;
"""
cursor.execute(cleanup_sql)
conn.commit()

# ---- Таблиця customers ----
cursor.execute("""
    CREATE TABLE customers (
        -- SERIAL = sequence + INTEGER NOT NULL + DEFAULT nextval()
        -- GENERATED ALWAYS AS IDENTITY — сучасна альтернатива SERIAL
        customer_id  INTEGER     GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        full_name    TEXT        NOT NULL,
        email        TEXT        NOT NULL UNIQUE,
        -- JSONB: зберігаємо гнучкі атрибути (адреса, теги, налаштування)
        metadata     JSONB       DEFAULT '{}',
        -- ARRAY: список тегів через нативний PostgreSQL масив
        tags         TEXT[]      DEFAULT '{}',
        created_at   TIMESTAMPTZ NOT NULL DEFAULT NOW()
    )
""")

# ---- Таблиця products ----
cursor.execute("""
    CREATE TABLE products (
        product_id   INTEGER     GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        name         TEXT        NOT NULL,
        category     TEXT        NOT NULL,
        -- NUMERIC(10,2) — точна десяткова арифметика для грошей
        -- REAL/FLOAT небезпечні для валют через помилки округлення!
        price        NUMERIC(10, 2) NOT NULL CHECK (price > 0),
        stock        INTEGER     NOT NULL DEFAULT 0,
        -- TSVECTOR: попередньо обчислений вектор для full-text search
        -- GENERATED ALWAYS AS: обчислюється автоматично при INSERT/UPDATE
        search_vec   TSVECTOR    GENERATED ALWAYS AS
            (to_tsvector('ukrainian', name || ' ' || category)) STORED
    )
""")

# ---- Таблиця orders ----
cursor.execute("""
    CREATE TABLE orders (
        order_id     INTEGER     GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        customer_id  INTEGER     NOT NULL REFERENCES customers(customer_id) ON DELETE RESTRICT,
        status       TEXT        NOT NULL DEFAULT 'pending'
                                 CHECK (status IN ('pending', 'confirmed', 'shipped', 'done', 'cancelled')),
        total_amount NUMERIC(12, 2) DEFAULT 0,
        ordered_at   TIMESTAMPTZ NOT NULL DEFAULT NOW()
    )
""")

# ---- Таблиця order_items ----
cursor.execute("""
    CREATE TABLE order_items (
        item_id      INTEGER     GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        order_id     INTEGER     NOT NULL REFERENCES orders(order_id) ON DELETE CASCADE,
        product_id   INTEGER     NOT NULL REFERENCES products(product_id),
        quantity     INTEGER     NOT NULL CHECK (quantity > 0),
        unit_price   NUMERIC(10, 2) NOT NULL
    )
""")

conn.commit()
print('Схему створено: customers, products, orders, order_items')

---
## 5. DML — Нові можливості PostgreSQL

### RETURNING — отримуємо дані одразу після INSERT

In [ ]:
# ---- INSERT з RETURNING — унікальна можливість PostgreSQL ----
# Дозволяє отримати будь-які колонки одразу після вставки
# (наприклад, згенерований ID або DEFAULT значення)
# В SQLite ми б використовували cursor.lastrowid

customers_data = [
    ('Олег Коваль',     'o.koval@shop.ua',
     {'city': 'Київ', 'tier': 'gold'},
     ['постійний', 'vip']),
    ('Марія Шевченко',  'm.shevchenko@shop.ua',
     {'city': 'Харків', 'tier': 'silver'},
     ['новий']),
    ('Дмитро Бондар',   'd.bondar@shop.ua',
     {'city': 'Львів', 'tier': 'bronze'},
     ['постійний']),
    ('Ірина Мельник',   'i.melnyk@shop.ua',
     {'city': 'Одеса', 'tier': 'gold'},
     ['vip', 'преміум']),
]

inserted_customers = []
for name, email, meta, tags in customers_data:
    # Json() — psycopg2 адаптер для автоматичної серіалізації dict → JSONB
    cursor.execute("""
        INSERT INTO customers (full_name, email, metadata, tags)
        VALUES (%s, %s, %s, %s)
        RETURNING customer_id, full_name, created_at
    """, (name, email, Json(meta), tags))

    # fetchone() повертає вставлений рядок одразу — без додаткового SELECT
    row = cursor.fetchone()
    inserted_customers.append(row)
    print(f'  Додано: {row["full_name"]} (id={row["customer_id"]}, at={row["created_at"]})')

conn.commit()

In [ ]:
# ---- Заповнюємо products ----
products_data = [
    ('MacBook Pro M3',    'ноутбуки',    89999.00, 15),
    ('iPhone 15 Pro',     'смартфони',   42999.00, 30),
    ('AirPods Pro',       'аудіо',        9999.00, 50),
    ('iPad Air',          'планшети',    29999.00, 20),
    ('Apple Watch Ultra', 'годинники',   27999.00, 12),
    ('Magic Keyboard',    'аксесуари',    5499.00, 45),
    ('Samsung Galaxy S24','смартфони',   38999.00, 25),
    ('Sony WH-1000XM5',   'аудіо',        12999.00, 35),
]

# executemany з psycopg2 — ефективне масове вставлення
cursor.executemany("""
    INSERT INTO products (name, category, price, stock)
    VALUES (%s, %s, %s, %s)
""", products_data)

# ---- INSERT ... ON CONFLICT DO UPDATE (UPSERT) ----
# Якщо email вже існує — оновлюємо metadata замість помилки
# excluded.* — посилання на рядок, який намагалися вставити
cursor.execute("""
    INSERT INTO customers (full_name, email, metadata)
    VALUES (%s, %s, %s)
    ON CONFLICT (email)
    DO UPDATE SET
        metadata   = EXCLUDED.metadata,
        full_name  = EXCLUDED.full_name
    RETURNING customer_id, full_name
""", ('Олег Коваль-ОНОВЛЕНО', 'o.koval@shop.ua', Json({'city': 'Київ', 'tier': 'platinum'})))

upsert_result = cursor.fetchone()
print(f'UPSERT: {upsert_result["full_name"]} (id={upsert_result["customer_id"]})')

conn.commit()
print('Products додано!')

In [ ]:
# ---- Додаємо замовлення ----
# Симулюємо реальну транзакцію: create order + add items + update total

def create_order(conn, customer_id: int, items: list[dict]) -> int:
    """
    Атомарне створення замовлення.
    items: [{'product_id': 1, 'quantity': 2}, ...]
    Повертає order_id.
    """
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        # Крок 1: Створюємо замовлення
        cur.execute("""
            INSERT INTO orders (customer_id, status)
            VALUES (%s, 'pending')
            RETURNING order_id
        """, (customer_id,))
        order_id = cur.fetchone()['order_id']

        total = 0
        for item in items:
            # Крок 2: Отримуємо ціну та перевіряємо наявність
            # FOR UPDATE блокує рядок від паралельних змін до кінця транзакції
            cur.execute("""
                SELECT price, stock FROM products
                WHERE product_id = %s
                FOR UPDATE
            """, (item['product_id'],))
            product = cur.fetchone()

            if not product:
                raise ValueError(f'Продукт {item["product_id"]} не знайдено')
            if product['stock'] < item['quantity']:
                raise ValueError(f'Недостатньо товару (є {product["stock"]}, потрібно {item["quantity"]})')

            # Крок 3: Вставляємо позицію
            cur.execute("""
                INSERT INTO order_items (order_id, product_id, quantity, unit_price)
                VALUES (%s, %s, %s, %s)
            """, (order_id, item['product_id'], item['quantity'], product['price']))

            # Крок 4: Зменшуємо залишок
            cur.execute("""
                UPDATE products SET stock = stock - %s WHERE product_id = %s
            """, (item['quantity'], item['product_id']))

            total += float(product['price']) * item['quantity']

        # Крок 5: Оновлюємо суму замовлення
        cur.execute("""
            UPDATE orders SET total_amount = %s, status = 'confirmed'
            WHERE order_id = %s
        """, (round(total, 2), order_id))

        conn.commit()
        print(f'  ✓ Замовлення #{order_id}: {len(items)} позицій, сума {total:.2f} грн')
        return order_id

    except Exception as e:
        conn.rollback()
        print(f'  ✗ Помилка: {e} — ROLLBACK')
        return -1


print('Створюємо замовлення:')
o1 = create_order(conn, 1, [{'product_id': 1, 'quantity': 1}, {'product_id': 3, 'quantity': 2}])
o2 = create_order(conn, 2, [{'product_id': 2, 'quantity': 1}])
o3 = create_order(conn, 1, [{'product_id': 4, 'quantity': 1}, {'product_id': 6, 'quantity': 3}])
o4 = create_order(conn, 4, [{'product_id': 5, 'quantity': 1}])
# Помилка: надто багато товару
o5 = create_order(conn, 3, [{'product_id': 1, 'quantity': 9999}])

---
## 6. EXPLAIN ANALYZE — бачимо план виконання

### Що показує EXPLAIN?

```
EXPLAIN ANALYZE SELECT ...
```

| Поле | Значення |
|------|----------|
| `cost=0.00..10.50` | Оціночна вартість: `startup..total` (в абстрактних одиницях) |
| `rows=100` | Оцінювана кількість рядків |
| `actual time=0.012..1.234` | Реальний час виконання (мс) |
| `loops=1` | Скільки разів виконувався цей вузол |
| `Seq Scan` | Послідовне сканування всієї таблиці |
| `Index Scan` | Сканування через B-Tree індекс |
| `Hash Join` | З'єднання через хеш-таблицю |
| `Nested Loop` | Вкладений цикл (одна таблиця мала) |

In [ ]:
def explain(conn, query, params=None, analyze=True):
    """Виконує EXPLAIN ANALYZE і красиво виводить план."""
    prefix = 'EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)' if analyze else 'EXPLAIN'
    cur = conn.cursor()
    cur.execute(f'{prefix} {query}', params)
    # EXPLAIN не змінює дані, але треба rollback щоб не лишати відкриту транзакцію
    plan = '\n'.join(row[0] for row in cur.fetchall())
    conn.rollback()
    print(plan)
    print()


# ---- Seq Scan — без індексу, повне сканування ----
print('=== Seq Scan (без індексу) ===')
explain(conn, """
    SELECT * FROM products WHERE category = 'смартфони'
""")

In [ ]:
# ---- Створюємо індекси ----

# B-Tree індекс на category — прискорює equality та range запити
cursor.execute('CREATE INDEX IF NOT EXISTS idx_products_category ON products(category)')

# B-Tree індекс на customer_id в orders — прискорює JOIN та WHERE
cursor.execute('CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(customer_id)')

# GIN індекс на TSVECTOR — для full-text search
cursor.execute('CREATE INDEX IF NOT EXISTS idx_products_fts ON products USING GIN(search_vec)')

# GIN індекс на JSONB metadata — для запитів всередині JSON
cursor.execute('CREATE INDEX IF NOT EXISTS idx_customers_meta ON customers USING GIN(metadata)')

# GIN індекс на ARRAY tags
cursor.execute('CREATE INDEX IF NOT EXISTS idx_customers_tags ON customers USING GIN(tags)')

conn.commit()
print('Індекси створено!')

# ---- Index Scan — той самий запит, але з індексом ----
print('\n=== Index Scan (після створення індексу) ===')
explain(conn, """
    SELECT * FROM products WHERE category = 'смартфони'
""")

In [ ]:
# ---- Hash Join vs Nested Loop ----
print('=== JOIN: Hash Join або Nested Loop ===')
explain(conn, """
    SELECT c.full_name, o.order_id, o.total_amount
    FROM customers c
        INNER JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.status = 'confirmed'
""")

---
## 7. JSONB — NoSQL всередині PostgreSQL

### JSONB vs JSON

| Тип | Зберігання | Індексація | Продуктивність |
|-----|-----------|------------|----------------|
| `JSON` | Текст (точна копія) | Тільки GIN на весь документ | Повільне читання |
| `JSONB` | Бінарний (de-serialized) | GIN на окремі ключі/значення | Швидке читання |

### Оператори JSONB

| Оператор | Значення | Приклад |
|----------|----------|---------|
| `->` | Повертає JSON | `metadata->'city'` → `"Київ"` |
| `->>` | Повертає TEXT | `metadata->>'city'` → `Київ` |
| `@>` | Містить | `metadata @> '{"tier":"gold"}'` |
| `?` | Ключ існує | `metadata ? 'phone'` |
| `#>` | Вкладений шлях | `data #> '{address,city}'` |

In [ ]:
# ---- JSONB запити ----

print('=== Запит всередині JSONB: клієнти з tier=gold або platinum ===')
cursor.execute("""
    SELECT
        full_name,
        metadata->>'city'  AS city,      -- ->> повертає text
        metadata->>'tier'  AS tier,
        tags
    FROM customers
    WHERE metadata->>'tier' IN ('gold', 'platinum')  -- фільтрація по JSONB
    ORDER BY full_name
""")
for row in cursor.fetchall():
    print(f'  {row["full_name"]}: {row["city"]}, tier={row["tier"]}, tags={row["tags"]}')

print()

# @> оператор: containment check (використовує GIN індекс!)
print('=== @> оператор: клієнти з {city: Київ} ===')
cursor.execute("""
    SELECT full_name, metadata
    FROM customers
    WHERE metadata @> %s
""", (Json({'city': 'Київ'}),))
for row in cursor.fetchall():
    print(f'  {row["full_name"]}: {row["metadata"]}')

print()

# Запит по масиву tags (ARRAY оператор: @> містить)
print('=== Клієнти з тегом vip (ARRAY containment) ===')
cursor.execute("""
    SELECT full_name, tags
    FROM customers
    WHERE tags @> ARRAY['vip']
""")
for row in cursor.fetchall():
    print(f'  {row["full_name"]}: {row["tags"]}')

In [ ]:
# ---- Full-Text Search (FTS) з GIN індексом ----
# Ми вже створили GIN індекс на search_vec (GENERATED ALWAYS AS)

print('=== Full-Text Search: пошук по "apple" ===')
cursor.execute("""
    SELECT
        name,
        category,
        price,
        -- ts_rank: числова релевантність (вищий = кращий збіг)
        ts_rank(search_vec, plainto_tsquery('ukrainian', %s)) AS rank
    FROM products
    WHERE search_vec @@ plainto_tsquery('ukrainian', %s)
    ORDER BY rank DESC
""", ('apple', 'apple'))

rows = cursor.fetchall()
if rows:
    for row in rows:
        print(f'  [{row["category"]}] {row["name"]} — {row["price"]} грн (rank={row["rank"]:.4f})')
else:
    # FTS залежить від мовного конфігу; спробуємо simple без стемінгу
    print('Немає результатів з ukrainian конфігурацією, спробуємо simple:')
    cursor.execute("""
        SELECT name, category, price
        FROM products
        WHERE to_tsvector('simple', name) @@ to_tsquery('simple', 'MacBook')
    """)
    for row in cursor.fetchall():
        print(f'  [{row["category"]}] {row["name"]} — {row["price"]} грн')

---
## 8. Window Functions — аналіз без GROUP BY

Window functions обчислюють результат **для кожного рядка** з урахуванням "вікна" (групи рядків), не згортаючи результат як `GROUP BY`.

```sql
функція() OVER (
    PARTITION BY колонка   -- ділимо на групи (аналог GROUP BY)
    ORDER BY колонка       -- сортування всередині вікна
    ROWS BETWEEN ...       -- межі вікна (за замовч. = вся партиція)
)
```

| Функція | Призначення |
|---------|-------------|
| `ROW_NUMBER()` | Унікальний номер рядка в партиції |
| `RANK()` | Рейтинг (з пропусками при однакових значеннях) |
| `DENSE_RANK()` | Рейтинг (без пропусків) |
| `LAG(col, n)` | Значення попереднього рядка |
| `LEAD(col, n)` | Значення наступного рядка |
| `SUM(...) OVER (...)` | Накопичувальна сума |
| `AVG(...) OVER (...)` | Ковзне середнє |

In [ ]:
# ---- Window Functions: рейтинг продуктів по категорії за ціною ----

print('=== RANK: топ продукти по ціні в кожній категорії ===')
cursor.execute("""
    SELECT
        category,
        name,
        price,
        -- RANK() всередині кожної категорії по ціні (спадання)
        RANK() OVER (
            PARTITION BY category
            ORDER BY price DESC
        ) AS price_rank,
        -- Порівнюємо ціну з середньою по категорії
        ROUND(price - AVG(price) OVER (PARTITION BY category), 2) AS vs_avg
    FROM products
    ORDER BY category, price_rank
""")

rows = cursor.fetchall()
print(f'{"Категорія":<15} {"Назва":<22} {"Ціна":<12} {"Ранг":<6} {"vs Avg"}')
print('-' * 68)
for row in rows:
    sign = '+' if float(row['vs_avg']) >= 0 else ''
    print(f'{row["category"]:<15} {row["name"]:<22} {float(row["price"]):<12.2f} '
          f'{row["price_rank"]:<6} {sign}{float(row["vs_avg"]):.2f}')

In [ ]:
# ---- Window Functions: LAG/LEAD та накопичувальна сума ----
# Аналіз замовлень по часу для кожного клієнта

print('=== LAG + накопичувальна сума замовлень ===')
cursor.execute("""
    SELECT
        c.full_name,
        o.order_id,
        o.total_amount,
        o.ordered_at::DATE AS order_date,
        -- Сума попереднього замовлення цього ж клієнта
        LAG(o.total_amount, 1, 0) OVER (
            PARTITION BY o.customer_id
            ORDER BY o.ordered_at
        ) AS prev_order_amount,
        -- Накопичувальна сума по клієнту
        SUM(o.total_amount) OVER (
            PARTITION BY o.customer_id
            ORDER BY o.ordered_at
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_total,
        -- Номер замовлення клієнта (1-ше, 2-ге, ...)
        ROW_NUMBER() OVER (
            PARTITION BY o.customer_id
            ORDER BY o.ordered_at
        ) AS order_seq
    FROM orders o
        INNER JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'confirmed'
    ORDER BY c.full_name, o.ordered_at
""")

rows = cursor.fetchall()
for row in rows:
    print(f'  {row["full_name"]:<22} #{row["order_seq"]}: '
          f'{float(row["total_amount"]):.0f} грн | '
          f'попередня={float(row["prev_order_amount"]):.0f} | '
          f'накоп.={float(row["cumulative_total"]):.0f}')

---
## 9. CTE (Common Table Expressions)

CTE — це іменований підзапит, визначений перед основним запитом через `WITH`.

**Переваги над підзапитами:**
- Читабельність: логічне розбиття запиту на кроки
- Повторне використання: один CTE можна використати кілька разів
- Рекурсія: через `WITH RECURSIVE` (дерева, графи)

In [ ]:
# ---- CTE: Аналіз продажів по категоріях ----

print('=== CTE: топ категорії за виручкою ===')
cursor.execute("""
    -- CTE 1: обчислюємо виручку на рівні позицій замовлень
    WITH item_revenue AS (
        SELECT
            p.category,
            p.name                      AS product_name,
            SUM(oi.quantity)            AS total_sold,
            SUM(oi.quantity * oi.unit_price) AS revenue
        FROM order_items oi
            INNER JOIN products p ON oi.product_id = p.product_id
            INNER JOIN orders o   ON oi.order_id = o.order_id
        WHERE o.status = 'confirmed'
        GROUP BY p.category, p.name
    ),
    -- CTE 2: підсумки по категорії + частка ринку
    category_totals AS (
        SELECT
            category,
            SUM(total_sold) AS units_sold,
            SUM(revenue)    AS total_revenue
        FROM item_revenue
        GROUP BY category
    )
    -- Фінальний запит: обидва CTE разом
    SELECT
        ct.category,
        ct.units_sold,
        ct.total_revenue,
        -- Частка від загальної виручки (підзапит по CTE)
        ROUND(
            ct.total_revenue * 100.0 /
            SUM(ct.total_revenue) OVER (),
        2) AS revenue_pct
    FROM category_totals ct
    ORDER BY ct.total_revenue DESC
""")

rows = cursor.fetchall()
for row in rows:
    print(f'  {row["category"]:<15}: {row["units_sold"]} шт., '
          f'{float(row["total_revenue"]):>12.2f} грн '
          f'({float(row["revenue_pct"]):.1f}%)')

In [ ]:
# ---- Рекурсивний CTE: обходимо ієрархію (дерево категорій) ----
# Рекурсивні CTE дозволяють обходити деревоподібні структури в SQL

# Спочатку створимо демонстраційну таблицю з ієрархією
cursor.execute('DROP TABLE IF EXISTS categories_tree')
cursor.execute("""
    CREATE TABLE categories_tree (
        cat_id   INTEGER PRIMARY KEY,
        name     TEXT NOT NULL,
        parent_id INTEGER REFERENCES categories_tree(cat_id)
    )
""")

cursor.executemany('INSERT INTO categories_tree VALUES (%s, %s, %s)', [
    (1, 'Електроніка',  None),
    (2, 'Комп\'ютери',  1),
    (3, 'Телефони',     1),
    (4, 'Ноутбуки',     2),
    (5, 'Настільні ПК', 2),
    (6, 'Смартфони',    3),
    (7, 'Кнопкові',     3),
])
conn.commit()

print('=== Рекурсивний CTE: обхід дерева категорій ===')
cursor.execute("""
    WITH RECURSIVE category_path AS (
        -- База рекурсії: кореневий вузол (без батька)
        SELECT cat_id, name, parent_id, 0 AS depth, name AS path
        FROM categories_tree
        WHERE parent_id IS NULL

        UNION ALL

        -- Рекурсивний крок: приєднуємо дочірні вузли
        SELECT c.cat_id, c.name, c.parent_id,
               cp.depth + 1,
               cp.path || ' → ' || c.name  -- будуємо шлях
        FROM categories_tree c
            INNER JOIN category_path cp ON c.parent_id = cp.cat_id
    )
    SELECT cat_id, depth, repeat('  ', depth) || name AS indented_name, path
    FROM category_path
    ORDER BY path
""")

for row in cursor.fetchall():
    print(f'  (id={row["cat_id"]}) {row["indented_name"]} — {row["path"]}')

---
## 10. Транзакції та рівні ізоляції

### Рівні ізоляції в PostgreSQL

| Рівень | Dirty Read | Non-repeatable Read | Phantom Read | Anomalies |
|--------|------------|--------------------|--------------|-----------|
| `READ UNCOMMITTED` | Можливий* | Можливий | Можливий | Всі |
| `READ COMMITTED` | Неможливий | Можливий | Можливий | Середні |
| `REPEATABLE READ` | Неможливий | Неможливий | Можливий* | Мінімальні |
| `SERIALIZABLE` | Неможливий | Неможливий | Неможливий | Жодних |

> *PostgreSQL реалізує READ UNCOMMITTED як READ COMMITTED (через MVCC)

**PostgreSQL за замовчуванням:** `READ COMMITTED`

In [ ]:
# ---- Демонстрація рівнів ізоляції ----
# Реальна демонстрація Non-repeatable Read потребує 2 паралельних підключень.
# Тут показуємо синтаксис та SERIALIZABLE транзакцію

# Відкриваємо друге з'єднання для паралельного запису
conn2 = psycopg2.connect(**DB_CONFIG)
conn2.autocommit = True  # всі зміни одразу видимі (без BEGIN)
cur2  = conn2.cursor(cursor_factory=RealDictCursor)

# ---- READ COMMITTED (за замовчуванням) ----
print('=== READ COMMITTED: кожен SELECT бачить нові committed дані ===')
conn.set_isolation_level(psycopg2.extensions.ISOLATION_LEVEL_READ_COMMITTED)
conn.autocommit = False

cursor.execute('BEGIN')

# Читаємо кількість замовлень
cursor.execute('SELECT COUNT(*) AS cnt FROM orders')
count_before = cursor.fetchone()['cnt']
print(f'  Читання 1: {count_before} замовлень')

# conn2 (autocommit) вставляє нове замовлення
cur2.execute("""
    INSERT INTO orders (customer_id, status, total_amount)
    VALUES (3, 'confirmed', 5000)
""")
print('  conn2: вставив нове замовлення (autocommit)')

# READ COMMITTED — другий SELECT бачить нові дані від conn2!
cursor.execute('SELECT COUNT(*) AS cnt FROM orders')
count_after = cursor.fetchone()['cnt']
print(f'  Читання 2 (READ COMMITTED): {count_after} замовлень — '
      f'{"Non-repeatable Read!" if count_after != count_before else "однаково"}')

conn.rollback()
conn2.close()
print()

In [ ]:
# ---- SERIALIZABLE: абсолютна ізоляція ----
print('=== SERIALIZABLE транзакція ===')
conn.autocommit = False

# Встановлюємо рівень ізоляції ПЕРЕД початком транзакції
cursor.execute('BEGIN ISOLATION LEVEL SERIALIZABLE')

cursor.execute('SELECT COUNT(*) AS cnt FROM orders')
count = cursor.fetchone()['cnt']
print(f'  Кількість замовлень: {count}')

# Виконуємо критичну бізнес-операцію з математичними гарантіями
cursor.execute("""
    UPDATE orders
    SET status = 'shipped'
    WHERE status = 'confirmed'
    RETURNING order_id, status
""")
updated = cursor.fetchall()
print(f'  Відправлено замовлень: {len(updated)}')

conn.commit()
print('  COMMIT — SERIALIZABLE гарантує відсутність будь-яких аномалій!')

---
## 11. Системні представлення PostgreSQL

PostgreSQL має потужні системні представлення (`pg_*`, `information_schema`) для моніторингу.

In [ ]:
# ---- Переглядаємо всі таблиці та їхні індекси ----

print('=== Таблиці в поточній БД ===')
cursor.execute("""
    SELECT tablename, tableowner
    FROM pg_tables
    WHERE schemaname = 'public'
    ORDER BY tablename
""")
for row in cursor.fetchall():
    print(f'  {row["tablename"]} (owner: {row["tableowner"]})')

print()
print('=== Індекси в БД ===')
cursor.execute("""
    SELECT indexname, tablename, indexdef
    FROM pg_indexes
    WHERE schemaname = 'public'
    ORDER BY tablename, indexname
""")
for row in cursor.fetchall():
    print(f'  [{row["tablename"]}] {row["indexname"]}')
    print(f'    {row["indexdef"]}')

print()
print('=== Розміри таблиць ===')
cursor.execute("""
    SELECT
        relname AS table_name,
        pg_size_pretty(pg_total_relation_size(relid)) AS total_size,
        n_live_tup  AS live_rows,
        n_dead_tup  AS dead_rows   -- рядки-кандидати для VACUUM
    FROM pg_stat_user_tables
    ORDER BY pg_total_relation_size(relid) DESC
""")
for row in cursor.fetchall():
    print(f'  {row["table_name"]:<20} {row["total_size"]:<12} '
          f'живих={row["live_rows"]} мертвих={row["dead_rows"]}')

In [ ]:
# Закриваємо з'єднання після роботи
cursor.close()
conn.close()
print('З\'єднання закрито!')

---
## Підсумок

| Концепція | Ключовий висновок |
|-----------|------------------|
| **psycopg2** | `RealDictCursor` для dict-результатів; `Json()` для JSONB |
| **RETURNING** | INSERT/UPDATE з негайним результатом — без зайвого SELECT |
| **UPSERT** | `INSERT ... ON CONFLICT DO UPDATE` — ідемпотентні операції |
| **EXPLAIN ANALYZE** | Показує реальний план виконання + час — обов'язковий для оптимізації |
| **Індекси** | B-Tree для equality/range; GIN для JSONB/FTS/ARRAY |
| **JSONB** | NoSQL всередині PostgreSQL — з індексами та операторами |
| **Window Functions** | RANK, LAG, LEAD, SUM OVER — аналіз без GROUP BY |
| **CTE** | WITH ... AS () — читабельні складні запити; WITH RECURSIVE — дерева |
| **Ізоляція** | За замовч. READ COMMITTED; SERIALIZABLE — абсолютна ізоляція |
| **MVCC** | Читання не блокує запис; UPDATE = нова версія рядка |

